# Pokemon Source Profiling & Reconciliation

This notebook is exploratory. It inspects both raw feeds first, identifies schema and data-quality differences, standardizes them to a common schema, and then compares overlapping records before deciding how to reconcile them.


In [ ]:
import json
import pandas as pd
import numpy as np

pd.set_option("display.max_columns", 30)
pd.set_option("display.max_rows", 100)
pd.set_option("display.width", 160)

with open("pokemon-source-a.json", "r", encoding="utf-8") as f:
    source_a = json.load(f)

with open("pokemon-source-b.json", "r", encoding="utf-8") as f:
    source_b = json.load(f)

df_a = pd.DataFrame(source_a)
df_b = pd.DataFrame(source_b)

print("Source A shape:", df_a.shape)
print("Source B shape:", df_b.shape)

print("\nSource A sample:")
display(df_a.head(10))

print("\nSource B sample:")
display(df_b.head(10))

In [ ]:
print("Source A columns and dtypes:")
print(df_a.dtypes)

print("\nSource B columns and dtypes:")
print(df_b.dtypes)

print("\nSource A null counts:")
display(df_a.isna().sum().to_frame("null_count"))

print("\nSource B null counts:")
display(df_b.isna().sum().to_frame("null_count"))

## Basic structural checks


In [ ]:
print("=== Source A: duplicate / key checks ===")
print("Duplicate full rows:", df_a.duplicated().sum())
print("Duplicate names:", df_a["name"].duplicated().sum())
print("Duplicate IDs:", df_a["id"].duplicated().sum())

print("\nRepeated IDs in Source A:")
display(df_a[df_a["id"].duplicated(keep=False)][["id", "name"]].sort_values(["id", "name"]))

print("\n=== Source B: duplicate / key checks ===")
print("Duplicate full rows:", df_b.duplicated().sum())
print("Duplicate names:", df_b["pokemonName"].duplicated().sum())
print("Duplicate IDs:", df_b["pokemonId"].duplicated().sum())

print("\nExact duplicate payloads in Source B:")
display(df_b[df_b.duplicated(keep=False)].sort_values(["pokemonName", "pokemonId"]))

print("\nRepeated IDs in Source B:")
display(df_b[df_b["pokemonId"].duplicated(keep=False)][["pokemonId", "pokemonName"]].sort_values(["pokemonId", "pokemonName"]))

## Inspect Pokemon forms before choosing a reconciliation key

The source IDs are not sufficient as a unique key because different forms can share a Pokedex ID.

The names also contain form information, such as Mega forms. This is inspected rather than stripped or transformed because the assessment does not define an entity-resolution rule for collapsing forms.


In [ ]:
print("Source A records containing 'Mega':")
display(df_a[df_a["name"].str.contains("Mega", case=False, na=False)][["id", "name", "type_1", "type_2"]])

print("\nSource B records containing 'Mega':")
display(df_b[df_b["pokemonName"].str.contains("Mega", case=False, na=False)][["pokemonId", "pokemonName", "primaryType", "secondaryType"]])

In [ ]:
source_a_types = set(
    pd.concat([df_a["type_1"], df_a["type_2"]], ignore_index=True)
    .dropna()
    .astype(str)
    .sort_values()
)

source_b_types = set(
    pd.concat([df_b["primaryType"], df_b["secondaryType"]], ignore_index=True)
    .dropna()
    .astype(str)
    .sort_values()
)

print("Source A unique type values:")
display(pd.DataFrame(sorted(source_a_types), columns=["source_a_type"]))

print("\nSource B unique type values:")
display(pd.DataFrame(sorted(source_b_types), columns=["source_b_type"]))

print("\nValues appearing in Source A but not Source B:")
display(pd.DataFrame(sorted(source_a_types - source_b_types), columns=["source_a_only"]))

print("\nValues appearing in Source B but not Source A:")
display(pd.DataFrame(sorted(source_b_types - source_a_types), columns=["source_b_only"]))

## Compare exact common keys before normalization

For reconciliation, use the supplied `pokemon_name` as the natural/business key for this exercise.

This first comparison deliberately uses the raw values. The purpose is to see what differs before deciding which differences are only representation differences.


In [ ]:
a_for_compare = df_a.rename(columns={"name": "pokemon_name"})
b_for_compare = df_b.rename(columns={"pokemonName": "pokemon_name"})

overlap = a_for_compare.merge(
    b_for_compare,
    on="pokemon_name",
    how="inner",
    suffixes=("_a", "_b"),
)

print("Exact pokemon_name overlap:", len(overlap))

print("\nCommon records:")
display(overlap[["pokemon_name"]].sort_values("pokemon_name").reset_index(drop=True))

In [ ]:
# Generic raw comparison: check every corresponding business field for exact
# value differences across records whose pokemon_name matches.

raw_column_pairs = {
    "pokemon_id": ("id", "pokemonId"),
    "primary_type": ("type_1", "primaryType"),
    "secondary_type": ("type_2", "secondaryType"),
    "total_stats": ("total", "totalStats"),
    "hp": ("hp_a", "hp_b"),
    "attack": ("attack_a", "attack_b"),
    "defense": ("defense_a", "defense_b"),
    "special_attack": ("sp_atk", "specialAttack"),
    "special_defense": ("sp_def", "specialDefense"),
    "speed": ("speed_a", "speed_b"),
    "generation": ("generation_a", "generation_b"),
    "is_legendary": ("legendary", "isLegendary"),
}

raw_differences = []

for _, row in overlap.iterrows():
    for canonical_name, (column_a, column_b) in raw_column_pairs.items():
        value_a = row[column_a]
        value_b = row[column_b]

        if pd.isna(value_a) and pd.isna(value_b):
            continue

        if pd.isna(value_a) or pd.isna(value_b) or str(value_a) != str(value_b):
            raw_differences.append({
                "pokemon_name": row["pokemon_name"],
                "column": canonical_name,
                "source_a_value": value_a,
                "source_b_value": value_b,
            })

raw_differences = pd.DataFrame(raw_differences)

print("Raw column-level differences across exact common keys:")

if raw_differences.empty:
    print("No raw differences found.")
else:
    display(raw_differences)
    print("\nDifferences by column:")
    display(raw_differences["column"].value_counts().to_frame("difference_count"))


In [ ]:
# Focused view of categorical representation differences.
type_differences = raw_differences[
    raw_differences["column"].isin(["primary_type", "secondary_type"])
]

print("Raw type representation differences:")
display(type_differences)

In [ ]:
# Investigate records that did not match on the exact business key.
# Exploratory only: no names are changed or collapsed.

a_names = set(df_a["name"].dropna().astype(str))
b_names = set(df_b["pokemonName"].dropna().astype(str))

a_only = df_a[~df_a["name"].isin(b_names)].copy()
b_only = df_b[~df_b["pokemonName"].isin(a_names)].copy()

print("Records exclusive to Source A by exact name:", len(a_only))
print("Records exclusive to Source B by exact name:", len(b_only))

In [ ]:
print("\nSource B exclusive records containing 'Sharpedo':")
display(b_only[b_only["pokemonName"].str.contains("Sharpedo", case=False, na=False)][["pokemonId", "pokemonName", "primaryType", "secondaryType"]])

In [ ]:
print("\nSource A exclusive records containing 'Mega':")
display(a_only[a_only["name"].str.contains("Mega", case=False, na=False)][["id", "name", "type_1", "type_2"]])

print("\nSource B exclusive records containing 'Mega':")
display(b_only[b_only["pokemonName"].str.contains("Mega", case=False, na=False)][["pokemonId", "pokemonName", "primaryType", "secondaryType"]])

## Standardize both feeds to one schema

The source-specific column names and representations are converted into the same canonical schema.

For Pokémon types, casing is not semantically meaningful, so values are trimmed and lowercased. Form names are not modified.


In [ ]:
def standardize_source_a(df):
    return pd.DataFrame({
        "pokemon_id": pd.to_numeric(df["id"], errors="coerce"),
        "pokemon_name": df["name"].astype("string").str.strip(),
        "primary_type": df["type_1"].astype("string").str.strip().str.lower(),
        "secondary_type": df["type_2"].astype("string").str.strip().str.lower(),
        "total_stats": pd.to_numeric(df["total"], errors="coerce"),
        "hp": pd.to_numeric(df["hp"], errors="coerce"),
        "attack": pd.to_numeric(df["attack"], errors="coerce"),
        "defense": pd.to_numeric(df["defense"], errors="coerce"),
        "special_attack": pd.to_numeric(df["sp_atk"], errors="coerce"),
        "special_defense": pd.to_numeric(df["sp_def"], errors="coerce"),
        "speed": pd.to_numeric(df["speed"], errors="coerce"),
        "generation": pd.to_numeric(df["generation"], errors="coerce"),
        "is_legendary": df["legendary"].astype("string").str.strip().str.lower().map({"true": True, "false": False}).astype("boolean"),
    })

def standardize_source_b(df):
    return pd.DataFrame({
        "pokemon_id": pd.to_numeric(df["pokemonId"], errors="coerce"),
        "pokemon_name": df["pokemonName"].astype("string").str.strip(),
        "primary_type": df["primaryType"].astype("string").str.strip().str.lower(),
        "secondary_type": df["secondaryType"].astype("string").str.strip().str.lower(),
        "total_stats": pd.to_numeric(df["totalStats"], errors="coerce"),
        "hp": pd.to_numeric(df["hp"], errors="coerce"),
        "attack": pd.to_numeric(df["attack"], errors="coerce"),
        "defense": pd.to_numeric(df["defense"], errors="coerce"),
        "special_attack": pd.to_numeric(df["specialAttack"], errors="coerce"),
        "special_defense": pd.to_numeric(df["specialDefense"], errors="coerce"),
        "speed": pd.to_numeric(df["speed"], errors="coerce"),
        "generation": pd.to_numeric(df["generation"], errors="coerce"),
        "is_legendary": df["isLegendary"].astype("boolean"),
    })

stg_a = standardize_source_a(df_a).drop_duplicates()
stg_b = standardize_source_b(df_b).drop_duplicates()

print("Standardized Source A:")
print(stg_a.dtypes)
display(stg_a.head(10))

print("\nStandardized Source B:")
print(stg_b.dtypes)
display(stg_b.head(10))

print("\nRows after exact duplicate removal:")
print("Source A:", len(stg_a))
print("Source B:", len(stg_b))

## Compare the normalized schemas

Now compare the same common records again after standardization.

This separates representation differences, such as `Ice` versus `ice`, from actual differences in the normalized data.


In [ ]:
normalized_overlap = stg_a.merge(
    stg_b,
    on="pokemon_name",
    how="inner",
    suffixes=("_a", "_b"),
)

normalized_columns = [
    "pokemon_id", "primary_type", "secondary_type", "total_stats",
    "hp", "attack", "defense", "special_attack", "special_defense",
    "speed", "generation", "is_legendary",
]

normalized_differences = []

for _, row in normalized_overlap.iterrows():
    for column in normalized_columns:
        value_a = row[f"{column}_a"]
        value_b = row[f"{column}_b"]

        if pd.isna(value_a) and pd.isna(value_b):
            continue

        if pd.isna(value_a) or pd.isna(value_b) or value_a != value_b:
            normalized_differences.append({
                "pokemon_name": row["pokemon_name"],
                "column": column,
                "source_a_value": value_a,
                "source_b_value": value_b,
            })

normalized_differences = pd.DataFrame(normalized_differences)

print("Normalized overlapping records:", len(normalized_overlap))
print("\nRemaining normalized differences:")

if normalized_differences.empty:
    print("No differences found across overlapping records.")
else:
    display(normalized_differences)

In [ ]:
print("=== Total-stat consistency ===")

def find_bad_totals(df):
    expected = (
        df["hp"] + df["attack"] + df["defense"]
        + df["special_attack"] + df["special_defense"] + df["speed"]
    )
    return df[df["total_stats"] != expected][
        ["pokemon_name", "total_stats", "hp", "attack", "defense",
         "special_attack", "special_defense", "speed"]
    ]

bad_a = find_bad_totals(stg_a)
bad_b = find_bad_totals(stg_b)

print("Source A invalid totals:", len(bad_a))
print("Source B invalid totals:", len(bad_b))

if len(bad_a): display(bad_a)
if len(bad_b): display(bad_b)

print("\n=== Final key counts ===")
print("Source A unique names:", stg_a["pokemon_name"].nunique())
print("Source B unique names:", stg_b["pokemon_name"].nunique())
print("Combined unique names:", pd.concat([stg_a["pokemon_name"], stg_b["pokemon_name"]]).nunique())

## Reconciliation conclusion

The profiling supports the following staging decisions:

- `pokemon_name` is used as the business key for reconciliation in this assessment, as it is unique within each source and distinguishes different Pokémon forms.
- Exact duplicate payloads are removed within each source.
- Source-specific fields are mapped into a common schema.
- Numeric and boolean values are cast to consistent types.
- Pokémon type values are trimmed and lowercased because casing is not semantically meaningful.
- Form names, including Mega forms, are preserved rather than collapsed because the assessment does not define an entity-resolution rule.
- Overlapping records are compared after normalization to identify cross-source conflicts.
- Since the assessment does not define source authority, deterministic source ordering is used as a fallback during final reconciliation. In production, source precedence would be agreed with the relevant data owners.
